In [3]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")
 
# ── ใส่ FRED API Key ตรงนี้
FRED_API_KEY = "e2f0c957d00e077c2b4a335a206cf2ff"   # ← เปลี่ยนตรงนี้
 
# ── Path
BASE       = "../../../data/processed/splits/"
FEAT_PATH  = "../../../data/processed/featured/featured_data.csv"
OUT_PATH   = "../../../data/processed/featured/featured_data_v2.csv"
 
# ── Date range (ให้ครอบคลุม raw data ทั้งหมด)
START = "2015-01-01"
END   = "2026-04-01"
 
# ================================================================================
# 0 : INSTALL CHECK
# ================================================================================
try:
    from fredapi import Fred
except ImportError:
    raise ImportError(
        "\n fredapi ยังไม่ได้ติดตั้ง\n"
        " รัน: pip install fredapi\n"
    )
 
try:
    import yfinance as yf
except ImportError:
    raise ImportError(
        "\n yfinance ยังไม่ได้ติดตั้ง\n"
        " รัน: pip install yfinance\n"
    )
 
fred = Fred(api_key=FRED_API_KEY)
print("FRED connection OK")
 
# ================================================================================
# 1 : LOAD BASE DATA
# ================================================================================
print("\n1 : โหลด featured_data เดิม...")
df = pd.read_csv("../data/processed/featured/featured_data.csv", parse_dates=["Date"], index_col="Date")
df.sort_index(inplace=True)
print(f"   Shape: {df.shape}  |  {df.index.min().date()} → {df.index.max().date()}")
 
# ── เตรียม date index สำหรับ merge (เฉพาะ business days)
bdays = pd.bdate_range(start=df.index.min(), end=df.index.max())
 
 
def to_bday_series(raw_series: pd.Series) -> pd.Series:
    """
    แปลง FRED series (monthly/weekly) → daily business day series
    ด้วย forward fill เพื่อไม่ให้เกิด look-ahead bias
    """
    s = raw_series.copy()
    s.index = pd.to_datetime(s.index)
    s = s.reindex(bdays, method="ffill")
    return s
 
 
# ================================================================================
# 2 : GROUP 1 — MACRO REGIME FEATURES
# ================================================================================
print("\n2 : ดาวน์โหลด Macro Regime features จาก FRED...")
 
# ── 2.1 Fed Funds Rate
print("   FEDFUNDS...")
fed_raw   = fred.get_series("FEDFUNDS", observation_start=START, observation_end=END)
fed_daily = to_bday_series(fed_raw)
# FRED release lag ~15 วัน → shift เพิ่มเพื่อ simulate real-world latency
fed_daily = fed_daily.shift(15)
 
df["f_fed_rate"]        = fed_daily.reindex(df.index)
df["f_fed_rate_chg_3m"] = fed_daily.diff(63).reindex(df.index)   # เปลี่ยนแปลง 3 เดือน
df["f_fed_hike_cycle"]  = (fed_daily.diff(63) > 0).astype(int).reindex(df.index)
# 1 = กำลัง hike cycle, 0 = hold/cut
 
# ── 2.2 Yield Curve (2Y vs 10Y)
print("   DGS2 / DGS10...")
y2_raw    = fred.get_series("DGS2",  observation_start=START, observation_end=END)
y10_raw   = fred.get_series("DGS10", observation_start=START, observation_end=END)
y2_daily  = to_bday_series(y2_raw)
y10_daily = to_bday_series(y10_raw)
 
spread          = y10_daily - y2_daily   # positive = normal, negative = inverted
df["f_yield_curve"]       = spread.reindex(df.index)
df["f_yield_curve_slope"] = spread.diff(20).reindex(df.index)    # ทิศทาง 1 เดือน
df["f_inverted_curve"]    = (spread < 0).astype(int).reindex(df.index)
# inverted curve = recession signal → gold มักได้รับ safe-haven demand
 
# ── 2.3 Real Interest Rate (TIPS 10Y)
print("   DFII10 (Real Rate)...")
tips_raw   = fred.get_series("DFII10", observation_start=START, observation_end=END)
tips_daily = to_bday_series(tips_raw)
 
df["f_real_rate_10y"]     = tips_daily.reindex(df.index)
df["f_real_rate_chg_1m"]  = tips_daily.diff(21).reindex(df.index)
df["f_real_rate_negative"] = (tips_daily < 0).astype(int).reindex(df.index)
# Real rate < 0 → bullish gold (opportunity cost ของการถือ gold ต่ำมาก)
 
# ── 2.4 CPI YoY
print("   CPIAUCSL (CPI)...")
cpi_raw  = fred.get_series("CPIAUCSL", observation_start=START, observation_end=END)
cpi_yoy  = cpi_raw.pct_change(12) * 100              # YoY %
# CPI release lag ~2 สัปดาห์
cpi_daily = to_bday_series(cpi_yoy).shift(15)
 
df["f_cpi_yoy"]         = cpi_daily.reindex(df.index)
df["f_cpi_trend_3m"]    = cpi_daily.diff(63).reindex(df.index)   # inflation เร่งขึ้นหรือลง
df["f_high_inflation"]  = (cpi_daily > 4).astype(int).reindex(df.index)
# >4% = high inflation regime → gold เป็น hedge ที่น่าสนใจ
 
print("   Group 1 done")
 
 
# ================================================================================
# 3 : GROUP 2 — MARKET STRESS FEATURES
# ================================================================================
print("\n3 : สร้าง Market Stress features...")
 
# ── 3.1 VIX Regime (VIX มีอยู่แล้วใน f_vix_close)
vix = df["f_vix_close"]
 
df["f_vix_regime"] = pd.cut(
    vix,
    bins=[0, 15, 25, 35, 9999],
    labels=[0, 1, 2, 3]
).astype(float)
# 0=calm(<15), 1=normal(15-25), 2=elevated(25-35), 3=crisis(>35)
 
df["f_vix_spike"]     = (vix > vix.rolling(60).mean() * 1.5).astype(int)
df["f_vix_trend_20d"] = vix.diff(20)
df["f_vix_ma_ratio"]  = vix / vix.rolling(252).mean()   # VIX vs 1Y average
 
# ── 3.2 DXY Trend (DXY มีอยู่แล้ว)
dxy = df["f_dxy_close"] if "f_dxy_close" in df.columns else df.filter(like="dxy").iloc[:, 0]
 
df["f_dxy_trend_20d"]   = dxy.diff(20)
df["f_dxy_trend_60d"]   = dxy.diff(60)
df["f_dxy_above_ma200"] = (dxy > dxy.rolling(200).mean()).astype(int)
# DXY แข็ง → gold มักอ่อน (inverse relationship แบบ classic)
 
# ── 3.3 High Yield Spread (Credit Stress)
print("   BAMLH0A0HYM2EY (HY Spread)...")  
hy_raw   = fred.get_series("BAMLH0A0HYM2EY",   # ← เปลี่ยนตรงนี้
                            observation_start=START, 
                            observation_end=END)
hy_daily = to_bday_series(hy_raw)

df["f_hy_spread"]         = hy_daily.reindex(df.index)
df["f_hy_spread_chg_20d"] = hy_daily.diff(20).reindex(df.index)
df["f_credit_stress"]     = (
    hy_daily > hy_daily.rolling(252).quantile(0.75)
).astype(int).reindex(df.index)
# Credit stress สูง = risk-off = gold มักได้รับ safe-haven flow
 
print("   Group 2 done")
 
 
# ================================================================================
# 4 : GROUP 3 — GOLD MICROSTRUCTURE
# ================================================================================
print("\n4 : สร้าง Gold Microstructure features...")
 
gold_ret = df["f_gold_ret_lag1"] if "f_gold_ret_lag1" in df.columns else df.filter(like="gold_ret").iloc[:, 0]
 
# ── 4.1 Seasonality (cyclical encoding ไม่มี data leakage)
df["f_month_sin"] = np.sin(2 * np.pi * df.index.month / 12)
df["f_month_cos"] = np.cos(2 * np.pi * df.index.month / 12)
df["f_is_q1"]     = (df.index.month.isin([1, 2, 3])).astype(int)
df["f_is_q4"]     = (df.index.month.isin([10, 11, 12])).astype(int)
# Gold มัก strong ใน Q1 (Chinese New Year demand) และ Q4 (year-end safe haven)
 
# ── 4.2 Gold Momentum Regime (สร้างจาก feature ที่มีอยู่)
ret_1m = gold_ret.rolling(21).mean()
ret_3m = gold_ret.rolling(63).mean()
 
df["f_gold_momentum_regime"] = np.where(
    (ret_1m > 0) & (ret_3m > 0), 2,     # strong uptrend
    np.where(
        (ret_1m < 0) & (ret_3m < 0), 0, # downtrend
        1                                 # mixed/sideways
    )
)
 
# ── 4.3 Gold Volatility Regime
gold_vol = df["f_gold_vol_20d"] if "f_gold_vol_20d" in df.columns else gold_ret.rolling(20).std()
vol_median = gold_vol.rolling(252).median()
 
df["f_gold_vol_regime"] = np.where(
    gold_vol > vol_median * 1.5, 2,      # high vol
    np.where(gold_vol < vol_median * 0.7, 0, 1)  # low / normal
)
 
print("   Group 3 done")
 
 
# ================================================================================
# 5 : ANTI-LEAKAGE VERIFICATION
# ================================================================================
print("\n5 : ตรวจสอบ Anti-Leakage...")
 
new_feat_cols = [
    "f_fed_rate", "f_fed_rate_chg_3m", "f_fed_hike_cycle",
    "f_yield_curve", "f_yield_curve_slope", "f_inverted_curve",
    "f_real_rate_10y", "f_real_rate_chg_1m", "f_real_rate_negative",
    "f_cpi_yoy", "f_cpi_trend_3m", "f_high_inflation",
    "f_vix_regime", "f_vix_spike", "f_vix_trend_20d", "f_vix_ma_ratio",
    "f_dxy_trend_20d", "f_dxy_trend_60d", "f_dxy_above_ma200",
    "f_hy_spread", "f_hy_spread_chg_20d", "f_credit_stress",
    "f_month_sin", "f_month_cos", "f_is_q1", "f_is_q4",
    "f_gold_momentum_regime", "f_gold_vol_regime",
]
 
checks_pass = True
for col in new_feat_cols:
    if col not in df.columns:
        print(f"   MISSING: {col}")
        checks_pass = False
        continue
    nan_count = df[col].isnull().sum()
    # NaN ต้นๆ เป็น acceptable (rolling warmup period)
    nan_pct = nan_count / len(df) * 100
    status = "OK" if nan_pct < 10 else "WARN"
    print(f"   {status}  {col:<35} NaN: {nan_count:4d} ({nan_pct:.1f}%)")
 
if checks_pass:
    print("\n   ทุก feature ผ่านการตรวจสอบ")
else:
    print("\n   มี feature บางตัวที่ต้องตรวจสอบเพิ่มเติม")
 
 
# ================================================================================
# 6 : DROP WARMUP ROWS & SAVE
# ================================================================================
print("\n6 : Fill NaN และ Save...")

# features ที่มี NaN แค่ช่วง warmup → forward fill แล้ว backward fill หัว
df_v2 = df.copy()
df_v2[new_feat_cols] = (
    df_v2[new_feat_cols]
    .ffill()   # fill ช่องว่างกลางด้วย forward fill
    .bfill()   # fill NaN หัวสุด (ก่อน first valid) ด้วย backward fill
)

# ตรวจว่าไม่มี NaN เหลือ
remaining_nan = df_v2[new_feat_cols].isnull().sum().sum()
print(f"   NaN เหลือหลัง fill: {remaining_nan}")

print(f"   Shape: {df_v2.shape}")
print(f"   Date range: {df_v2.index.min().date()} → {df_v2.index.max().date()}")

df_v2.to_csv("../data/processed/featured/2_featured_data.csv")
print(f"   Saved → ../data/processed/featured/2_featured_data.csv")
 
 
# ================================================================================
# 7 : QUICK SUMMARY ของ New Features
# ================================================================================
print("\n" + "=" * 60)
print("  NEW FEATURES SUMMARY")
print("=" * 60)
 
groups = {
    "Macro Regime (Group 1)": [c for c in new_feat_cols if any(k in c for k in ["fed", "yield", "real_rate", "cpi", "inflation", "inverted"])],
    "Market Stress (Group 2)": [c for c in new_feat_cols if any(k in c for k in ["vix_regime", "vix_spike", "vix_trend", "vix_ma", "dxy_trend", "dxy_above", "hy_", "credit"])],
    "Gold Microstructure (Group 3)": [c for c in new_feat_cols if any(k in c for k in ["month", "is_q", "momentum_regime", "vol_regime"])],
}
 
for grp_name, cols in groups.items():
    print(f"\n  {grp_name}: {len(cols)} features")
    for col in cols:
        if col in df_v2.columns:
            val = df_v2[col].describe()
            print(f"    {col:<35} mean={val['mean']:>7.3f}  std={val['std']:>6.3f}")
 
print(f"""
  ขั้นตอนต่อไป:
  → รัน Step 5 (train_test_split) ใหม่บน featured_data_v2.csv
  → Retrain RF และเปรียบเทียบ DA/IC กับ baseline
""")












FRED connection OK

1 : โหลด featured_data เดิม...
   Shape: (2542, 57)  |  2016-02-16 → 2026-03-27

2 : ดาวน์โหลด Macro Regime features จาก FRED...
   FEDFUNDS...
   DGS2 / DGS10...
   DFII10 (Real Rate)...
   CPIAUCSL (CPI)...
   Group 1 done

3 : สร้าง Market Stress features...
   BAMLH0A0HYM2EY (HY Spread)...
   Group 2 done

4 : สร้าง Gold Microstructure features...
   Group 3 done

5 : ตรวจสอบ Anti-Leakage...
   OK  f_fed_rate                          NaN:   15 (0.6%)
   OK  f_fed_rate_chg_3m                   NaN:   76 (3.0%)
   OK  f_fed_hike_cycle                    NaN:    0 (0.0%)
   OK  f_yield_curve                       NaN:   17 (0.7%)
   OK  f_yield_curve_slope                 NaN:  138 (5.4%)
   OK  f_inverted_curve                    NaN:    0 (0.0%)
   OK  f_real_rate_10y                     NaN:   17 (0.7%)
   OK  f_real_rate_chg_1m                  NaN:  145 (5.7%)
   OK  f_real_rate_negative                NaN:    0 (0.0%)
   OK  f_cpi_yoy                         